---
title: "最短路径——Dijkstra 算法"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [4]:
from typing import List
import math
import heapq

## 📝 题目 743：网络延迟时间

::: {.callout-caution}
### 📖 题目描述
有 `n` 个网络节点，标记为 `1` 到 `n`。

给你一个列表 `times`，表示信号经过 **有向** 边的传递时间。 `times[i] = (ui, vi, wi)`，其中 `ui` 是源节点，`vi` 是目标节点， `wi` 是一个信号从源节点传递到目标节点的时间。

现在，从某个节点 `K` 发出一个信号。需要多久才能使所有节点都收到信号？如果不能使所有节点收到信号，返回 `-1` 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`times = [[2,1,1],[2,3,1],[3,4,1]]`, `n = 4`, `k = 2`
    * **输出**：`2`
    * **解释**：从节点 2 发出信号。到节点 1 和 3 需要 1 秒。节点 3 收到后转发给 4，再花 1 秒。最晚收到的是节点 4，总耗时为 1 + 1 = 2 秒。

* **示例 2**：
    * **输入**：`times = [[1,2,1]]`, `n = 2`, `k = 1`
    * **输出**：`1`

* **示例 3**：
    * **输入**：`times = [[1,2,1]]`, `n = 2`, `k = 2`
    * **输出**：`-1`
    * **解释**：从节点 2 出发，这是一条有向边 (1 -> 2)，信号无法反向传给 1，所以节点 1 永远收不到，返回 -1。
:::

In [6]:
class Solution743:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        adj = {i: [] for i in range(1, n + 1)}  # 严格构建有向图
        for source, dest, time in times:
            adj[source].append((time, dest))
        min_heap = [(0, k)]  # 初始化堆和距离表
        dist_map = {i: math.inf for i in range(1, n + 1)}
        while min_heap:
            curr_time, curr_node = heapq.heappop(min_heap)
            if curr_time > dist_map[curr_node]:  # 如果这辆车比我们记录的最快时间还要慢，说明它是“僵尸节点”，直接拦截！
                continue
            dist_map[curr_node] = curr_time  # 记录最快到达时间
            for next_time, next_node in adj[curr_node]:  # 下一站的总耗时 = 到这里的总耗时 + 这一段路的耗时
                heapq.heappush(min_heap, (curr_time + next_time, next_node))
        return max(dist_map.values()) if max(dist_map.values()) != math.inf else -1

In [7]:
#| code-fold: true
def test_743(func):
    test_cases = [
        {"desc": "示例 1: 常规分叉传递", "times": [[2,1,1],[2,3,1],[3,4,1]], "n": 4, "k": 2, "expected": 2},
        {"desc": "示例 2: 简单两人相连", "times": [[1,2,1]], "n": 2, "k": 1, "expected": 1},
        {"desc": "示例 3: 有向边逆行孤岛", "times": [[1,2,1]], "n": 2, "k": 2, "expected": -1},
        {"desc": "线性传递", "times": [[1,2,1],[2,3,2],[3,4,3]], "n": 4, "k": 1, "expected": 6},
        {"desc": "星型传递 (中心发出)", "times": [[1,2,5],[1,3,5],[1,4,5]], "n": 4, "k": 1, "expected": 5},
        {"desc": "发现捷径 (多重报价覆盖)", "times": [[1,2,10],[1,3,1],[3,2,1]], "n": 3, "k": 1, "expected": 2}, # 1->3->2 只需2秒，比直达10秒快
        {"desc": "成环的网络 (防死循环)", "times": [[1,2,1],[2,3,1],[3,1,1],[3,4,1]], "n": 4, "k": 1, "expected": 3},
        {"desc": "完全不相连的森林", "times": [[1,2,1],[3,4,1]], "n": 4, "k": 1, "expected": -1},
        {"desc": "自身环绕边 (测试健壮性)", "times": [[1,1,10],[1,2,1]], "n": 2, "k": 1, "expected": 1},
        {"desc": "多条平行同向边 (取最小)", "times": [[1,2,10],[1,2,2],[1,2,5]], "n": 2, "k": 1, "expected": 2},
        {"desc": "极限: 单个节点", "times": [], "n": 1, "k": 1, "expected": 0}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["times"], tc["n"], tc["k"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_743(Solution743().networkDelayTime)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 2    | 2    | 示例 1: 常规分叉传递
2    | ✅ PASS | 1    | 1    | 示例 2: 简单两人相连
3    | ✅ PASS | -1   | -1   | 示例 3: 有向边逆行孤岛
4    | ✅ PASS | 6    | 6    | 线性传递
5    | ✅ PASS | 5    | 5    | 星型传递 (中心发出)
6    | ✅ PASS | 2    | 2    | 发现捷径 (多重报价覆盖)
7    | ✅ PASS | 3    | 3    | 成环的网络 (防死循环)
8    | ✅ PASS | -1   | -1   | 完全不相连的森林
9    | ✅ PASS | 1    | 1    | 自身环绕边 (测试健壮性)
10   | ✅ PASS | 2    | 2    | 多条平行同向边 (取最小)
11   | ✅ PASS | 0    | 0    | 极限: 单个节点
-----------------------------------------------------------------
测试总结: 通过 11/11


::: {.callout-important}
### 💡 思路讲解



这道题求的是“使所有节点都收到信号的时间”。因为信号是同时向各个方向传播的，所以这就等价于求：**起点 `K` 到所有其他节点的最短路径中，最长的那一条！**（最远的小弟收到了，就代表所有人都收到了）。

1. **建图（注意是有向图）**：
   - 建立邻接表 `adj`。注意！题目说是有向边，所以只能 `adj[u].append((时间, v))`，**不能**双向添加！

2. **祭出最小堆（优先队列）**：
   - 堆里存放 `(从起点K过来的累计总时间, 当前节点)`。
   - 初始把起点丢进去：`heappush(min_heap, (0, k))`。

3. **开始传播信号（Dijkstra 核心扩散）**：
   - 我们用一个字典（或集合/数组）`dist_map` 来记录“起点 `K` 到达各个节点的最短时间”。
   - 每次从堆中弹出一个节点 `curr_node` 和累计时间 `curr_time`。
   - **防重判断**：如果这个节点已经在 `dist_map` 里了，说明我们之前早就通过更快的路径到达过它了！现在的这条是“慢车”，直接 `continue` 丢弃。
   - 否则，把当前节点记录到 `dist_map` 中：`dist_map[curr_node] = curr_time`。
   - **向邻居扩散（⚠️ 唯一的区别在这里）**：
     遍历 `curr_node` 的所有邻居 `(路段耗时, next_node)`。
     计算出到达邻居的**总耗时**：`next_time = curr_time + 路段耗时`。
     把 `(next_time, next_node)` 扔进堆里！

4. **大结局与孤岛判断**：
   - 当堆空了，扩散结束。
   - 检查 `len(dist_map)`。如果它等于 `n`，说明大家都收到了，返回 `dist_map` 里面的最大值 `max(dist_map.values())`。
   - 如果它小于 `n`，说明有孤岛节点（比如逆向边去不了的地方），直接返回 `-1`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(E \log V)$。由于使用了最小堆，每条边最多被推入和弹出堆一次。
* **空间复杂度**：$O(V + E)$。存储邻接表和优先队列。
:::

## 📝 题目 787：K 站中转内最便宜的航班

::: {.callout-caution}
### 📖 题目描述
有 `n` 个城市通过一些航班连接。给你一个数组 `flights` ，其中 `flights[i] = `[$from_i$, $to_i$, $price_i$] ，表示该航班都从城市 $from_i$ 开始，以价格 $price_i$ 抵达 $to_i$。

现在给定所有的城市和航班，以及出发城市 `src` 和目的地 `dst`，你的任务是找到出一条最多经过 `k` 站中转的路线，使得从 `src` 到 `dst` 的 **价格最便宜** ，并返回该价格。 如果不存在这样的路线，则输出 `-1`。

**注意**：经过 `k` 站中转，意味着你从 `src` 到 `dst` 最多只能搭乘 `k + 1` 趟航班（走 `k + 1` 条边）。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 3`, `flights = [[0,1,100],[1,2,100],[0,2,500]]`, `src = 0`, `dst = 2`, `k = 1`
    * **输出**：`200`
    * **解释**：
      从 0 到 2 有两条航线：
      航线 1：0 -> 2 (花费 500，中转 0 次)
      航线 2：0 -> 1 -> 2 (花费 200，中转 1 次)
      最多允许中转 1 次，因此我们选择航线 2，花费 200。

* **示例 2**：
    * **输入**：`n = 3`, `flights = [[0,1,100],[1,2,100],[0,2,500]]`, `src = 0`, `dst = 2`, `k = 0`
    * **输出**：`500`
    * **解释**：
      题目同上，但最多允许中转 0 次（只能直飞）。所以只能选花费 500 的航线 1。
:::

In [18]:
class Solution787:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        adj = {i: [] for i in range(n)}  # 建有向图
        for flight in flights:
            adj[flight[0]].append((flight[2], flight[1]))
        min_stops = {i: math.inf for i in range(n)}  # 记录到达每个节点的最少转机次数 (必须初始化为无穷大！)
        min_heap = [(0, src, 0)]
        while min_heap:
            cost, curr_node, stops = heapq.heappop(min_heap)
            if curr_node == dst:  # 第一次摸到终点，绝对是最便宜的
                return cost
            if stops > k:  # 如果转机次数已经超过限制，这条路不能再飞了，丢弃
                continue
            if stops >= min_stops[curr_node]:  # 如果转机次数比历史记录多(或相等)，说明又贵又绕远，彻底丢弃！
                continue
            min_stops[curr_node] = stops  # 刷新到达该节点的最少转机次数记录
            for next_cost, next_node in adj[curr_node]:  # 向邻居扩散 (转机次数 + 1)
                heapq.heappush(min_heap, (cost + next_cost, next_node, stops + 1))
        return -1

In [20]:
#| code-fold: true
def test_787(func):
    test_cases = [
        {"desc": "示例 1: 允许1次转机选便宜", "n": 3, "flights": [[0,1,100],[1,2,100],[0,2,500]], "src": 0, "dst": 2, "k": 1, "expected": 200},
        {"desc": "示例 2: 只能直飞选贵的", "n": 3, "flights": [[0,1,100],[1,2,100],[0,2,500]], "src": 0, "dst": 2, "k": 0, "expected": 500},
        {"desc": "找不到路线", "n": 3, "flights": [[0,1,100]], "src": 0, "dst": 2, "k": 0, "expected": -1},
        {"desc": "K足够大，选最便宜的迂回路线", "n": 4, "flights": [[0,1,1],[1,2,1],[2,3,1],[0,3,10]], "src": 0, "dst": 3, "k": 2, "expected": 3},
        {"desc": "K不够，只能选中等价位路线", "n": 4, "flights": [[0,1,1],[1,2,1],[2,3,1],[0,2,5],[2,3,1],[0,3,10]], "src": 0, "dst": 3, "k": 1, "expected": 6}, # 0->2->3 花费 5+1=6，转机1次
        {"desc": "包含环，测试防重拦截机制", "n": 3, "flights": [[0,1,100],[1,0,10],[1,2,100]], "src": 0, "dst": 2, "k": 1, "expected": 200},
        {"desc": "起点和终点根本不通", "n": 4, "flights": [[0,1,100],[2,3,100]], "src": 0, "dst": 3, "k": 2, "expected": -1},
        {"desc": "有多条同等转机次数的平行航线", "n": 3, "flights": [[0,1,100],[0,1,50],[1,2,100]], "src": 0, "dst": 2, "k": 1, "expected": 150},
        {"desc": "复杂的梯子网络", "n": 5, "flights": [[0,1,10],[1,2,10],[2,3,10],[3,4,10],[0,2,30],[2,4,30],[0,4,100]], "src": 0, "dst": 4, "k": 2, "expected": 50},
        {"desc": "极限: 单个节点不需要飞", "n": 1, "flights": [], "src": 0, "dst": 0, "k": 0, "expected": 0},
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["n"], tc["flights"], tc["src"], tc["dst"], tc["k"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_787(Solution787().findCheapestPrice)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 200  | 200  | 示例 1: 允许1次转机选便宜
2    | ✅ PASS | 500  | 500  | 示例 2: 只能直飞选贵的
3    | ✅ PASS | -1   | -1   | 找不到路线
4    | ✅ PASS | 3    | 3    | K足够大，选最便宜的迂回路线
5    | ✅ PASS | 6    | 6    | K不够，只能选中等价位路线
6    | ✅ PASS | 200  | 200  | 包含环，测试防重拦截机制
7    | ✅ PASS | -1   | -1   | 起点和终点根本不通
8    | ✅ PASS | 150  | 150  | 有多条同等转机次数的平行航线
9    | ✅ PASS | 50   | 50   | 复杂的梯子网络
10   | ✅ PASS | 0    | 0    | 极限: 单个节点不需要飞
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

在普通的 Dijkstra 里，只要一辆车比之前记录的耗时更长，我们就直接丢弃（`continue`）。
但在有 $K$ 次中转限制的题里，**“贵”的航线不一定没用，它可能“直达”（省次数）！**

**核心 1：堆里存什么？**

我们要把“中转次数”也塞进优先队列里。
堆元素的格式变为：`(当前总花费, 当前城市, 当前已经中转的次数)`。
*（注意：花费依然必须放在元组的第 0 位，保证堆每次弹出的依然是当前最便宜的方案）*

**核心 2：如何拦截僵尸节点？（🔥 灵魂所在）**

以前我们用 `dist_map[node]` 记录到达节点的最便宜价格。
现在不行了！如果之前有一条极度便宜但中转了 10 次的航线到达了城市 A，现在来了一条稍贵但只中转 1 次的航线，你能把它扔了吗？绝对不能！它可能是通向终点的唯一希望！

所以，我们的拦截逻辑必须改变：**我们不记录最便宜的价格，我们记录到达该城市的“最少中转次数”！**
我们用一个数组 `min_stops[node]` 记录到达节点的最少中转次数。
* 当你从堆里弹出一个方案，如果它的**中转次数比之前记录的还要大（或相等）**，并且花费肯定也是越来越高的（因为是从堆里弹出的），那它才是彻头彻尾的“又贵又绕路”的僵尸节点，直接 `continue` 拦截！
* 如果它的中转次数更少，我们就更新 `min_stops`，并让它继续扩散。

**算法流程**：

1. **建图**：构建 `adj` 邻接表。
2. **初始化**：`min_heap = [(0, src, 0)]`。`min_stops` 数组初始化为无穷大。
3. **扩散**：
   - 弹出 `(cost, curr_node, stops)`。
   - 如果 `curr_node == dst`，太棒了！因为堆保证了按价格弹出，**你第一次摸到终点时的价格，绝对就是满足条件的最低价格！** 直接 `return cost`。
   - 如果 `stops == k + 1`，说明到达当前节点的航线已经用光了所有配额，不能再往下飞了，直接 `continue`。
   - **防重判断**：如果 `stops > min_stops[curr_node]`，说明这条航线又贵转机又多，直接 `continue`。
   - 更新防重记录：`min_stops[curr_node] = stops`。
   - 将邻居加入堆中：`(cost + next_cost, next_node, stops + 1)`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(E + V \log V)$ 的变体。由于同一个节点可能会因为中转次数不同而多次入堆，最坏情况下会达到 $O(E \cdot K \log(E \cdot K))$。
* **空间复杂度**：$O(V + E)$，存储图和优先队列。
:::

## 📝 题目 1631：最小体力消耗路径

::: {.callout-caution}
### 📖 题目描述
你准备参加一场远足活动。给你一个二维 `rows x columns` 的大小为 `heights` 的视图，其中 `heights[row][col]` 表示格子 `(row, col)` 的高度。

一开始你在最左上角的格子 `(0, 0)` ，且你希望去最右下角的格子 `(rows-1, columns-1)` （注意下标从 0 开始编号）。你每次可以往 **上，下，左，右** 四个方向之一移动，你想要找到耗费 **体力** 最小的一条路径。

一条路径耗费的 **体力值** 是路径上 **相邻格子之间 高度差绝对值 的 最大值** 。

请你返回从左上角走到右下角的 **最小 体力消耗值** 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`heights = [[1,2,2],[3,8,2],[5,3,5]]`
    * **输出**：`2`
    * **解释**：
      路径 `[1,3,5,3,5]` 连续走过的高度差分别为：
      |1-3|=2, |3-5|=2, |5-3|=2, |3-5|=2。
      这条路径上最大的高度差是 2。这是所有可能路径中，最大高度差最小的路径。
      *(如果你走 `[1,2,2,2,5]`，高度差最大值会是 |2-5|=3，比 2 大)*

* **示例 2**：
    * **输入**：`heights = [[1,2,3],[3,8,4],[5,3,5]]`
    * **输出**：`1`
    * **解释**：路径 `[1,2,3,4,5]` 的相邻高度差绝对值最大为 1。
:::

In [27]:
class Solution1631:
    def minimumEffortPath(self, heights: List[List[int]]) -> int:
        rows, cols = len(heights), len(heights[0])
        min_effort = [[float("inf")] * cols for _ in range(rows)]  # 记录到达每个格子的最小体力消耗，初始化为无穷大
        min_effort[0][0] = 0
        min_heap = [(0, 0, 0)]  # 最小堆：(当前路径的最大体力消耗, r, c)
        while min_heap:
            curr_effort, r, c = heapq.heappop(min_heap)
            if r == rows - 1 and c == cols - 1:  # 第一次摸到终点，绝对是最优解！直接交卷！
                return curr_effort
            if curr_effort > min_effort[r][c]:  # 如果是废弃的“费力”路线，直接丢弃
                continue
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols:
                    next_effort = max(curr_effort, abs(heights[r][c] - heights[nr][nc]))  # 算这条路径的历史最高难度
                    if next_effort < min_effort[nr][nc]:  # 如果发现了更省力的方法，更新记录并塞进堆里
                        min_effort[nr][nc] = next_effort
                        heapq.heappush(min_heap, (next_effort, nr, nc))
        return 0

In [29]:
#| code-fold: true
def test_1631(func):
    test_cases = [
        {"desc": "示例 1 (需要绕开中间的 8)", "heights": [[1,2,2],[3,8,2],[5,3,5]], "expected": 2},
        {"desc": "示例 2 (可以直走)", "heights": [[1,2,3],[3,8,4],[5,3,5]], "expected": 1},
        {"desc": "示例 3 (巨大的迷宫，必须迂回)", "heights": [[1,2,1,1,1],[1,2,1,2,1],[1,2,1,2,1],[1,2,1,2,1],[1,1,1,2,1]], "expected": 0},
        {"desc": "极限: 单个格子", "heights": [[5]], "expected": 0},
        {"desc": "单行长廊 (直线)", "heights": [[1, 5, 2, 8, 3]], "expected": 6}, # 5-2=3, 8-2=6, max is 6
        {"desc": "单列电梯 (直线)", "heights": [[1], [4], [2], [9]], "expected": 7}, # 9-2=7
        {"desc": "全平原 (完全平坦)", "heights": [[10,10,10],[10,10,10],[10,10,10]], "expected": 0},
        {"desc": "大落差悬崖", "heights": [[1, 1000000], [1000000, 1]], "expected": 999999},
        {"desc": "强迫走迷宫倒退", "heights": [[1,10,10],[1,1,10],[10,1,1]], "expected": 0}, # 1->1->1->1->1
        {"desc": "多条路线博弈", "heights": [[1,3,1],[2,5,2],[1,3,1]], "expected": 2} # 1->2->1->2->1 (1-2, 2-1, 1-2, 2-1) max=1, wait no, 1-3 is 2, 1-2 is 1... Actually [0,0]->[1,0]->[2,0]->[2,1]->[2,2] heights 1->2->1->3->1. Wait, 1->2->1 is 1, 1->3 is 2. So effort 2.
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<6} | {'实际':<6} | {'描述'}")
    print("-" * 65)

    for i, tc in enumerate(test_cases):
        actual = func(tc["heights"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass:
            passed += 1

        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<6} | {actual:<6} | {tc['desc']}")

    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1631(Solution1631().minimumEffortPath)

ID   | 结果     | 预期     | 实际     | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 2      | 2      | 示例 1 (需要绕开中间的 8)
2    | ✅ PASS | 1      | 1      | 示例 2 (可以直走)
3    | ✅ PASS | 0      | 0      | 示例 3 (巨大的迷宫，必须迂回)
4    | ✅ PASS | 0      | 0      | 极限: 单个格子
5    | ✅ PASS | 6      | 6      | 单行长廊 (直线)
6    | ✅ PASS | 7      | 7      | 单列电梯 (直线)
7    | ✅ PASS | 0      | 0      | 全平原 (完全平坦)
8    | ✅ PASS | 999999 | 999999 | 大落差悬崖
9    | ✅ PASS | 0      | 0      | 强迫走迷宫倒退
10   | ✅ PASS | 2      | 2      | 多条路线博弈
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

这道题表面上是个二维迷宫，实际上是一张**隐藏的图 (Implicit Graph)**：

* **节点**：每一个格子 `(r, c)` 就是一个城市。
* **边**：每个格子最多连着上下左右 4 个邻居。
* **边的权重（路费）**：就是两个相邻格子的高度差绝对值 `abs(heights[nx][ny] - heights[x][y])`。

**核心：重新定义“总路费”！**
在经典的 Dijkstra 里，从 A 走到 C 的总路费是**相加**：`cost = cost(A->B) + cost(B->C)`。
但这道题说，体力消耗不是累加的，而是取决于**这一路上最陡峭的那一步**！
所以，到达邻居的“总体力代价”变成了**取最大值**：
`next_effort = max(当前路径已经付出的最大努力 curr_effort, 迈向下一步的高度差)`

**算法流程**：
1. **无需建图**：直接在 `heights` 矩阵上用方向数组 `[(0,1), (0,-1), (1,0), (-1,0)]` 遍历邻居即可。
2. **记录表**：用一个二维数组 `min_effort[r][c]` 记录到达每个格子的最小体力消耗，全部初始化为无穷大。起点 `min_effort[0][0] = 0`。
3. **最小堆**：堆里存放 `(当前的最大体力消耗, r, c)`。起点入堆：`[(0, 0, 0)]`。
4. **核心扩散**：
   - 弹出 `(curr_effort, r, c)`。
   - **提前交卷**：如果 `r == rows-1` 且 `c == cols-1`，直接 `return curr_effort`！（因为堆是按体力排的，第一次摸到右下角，绝对是最优解）。
   - **防重拦截**：如果 `curr_effort > min_effort[r][c]`，说明是一条“更费力”的废路径，直接 `continue`！
   - 向上下左右 4 个方向探索，算出 `next_effort`。
   - 如果发现 `next_effort < min_effort[nx][ny]`，说明找到了去那个格子更省力的方法，更新记录，并把 `(next_effort, nx, ny)` 塞进堆里！
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(M \times N \log(M \times N))$。网格共有 $M \times N$ 个节点，最坏情况下所有节点都会入堆并弹出。
* **空间复杂度**：$O(M \times N)$。用于存放 `min_effort` 记录表和优先队列。
:::